# Full Replication: Quantifying Investor Sentiment with Multimodal Data in the Chinese Stock Market

**Sample Period:** 2014-01-01 to 2026-03-20 (extended from paper's 2014-2024)

Replication targets:
1. Table 1 — Descriptive Statistics
2. Table 3 — PhotoPes Regression
3. Table 4 — TextPes Regression
4. Table 5 — Interaction Effect (PhotoPes + TextPes + Interaction)
5. Table 6 — Conditional Analysis (Extreme vs Non-Extreme PhotoPes)
6. Table 7 — Robustness Checks (No Winsorization / GARCH-adjusted / Remove Extreme Returns)
7. Portfolio Backtest + Factor Regression
8. Out-of-Sample Prediction

In [ ]:
import warnings, logging, os
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from scipy.stats.mstats import winsorize
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
plt.rcParams.update({"figure.figsize": (14, 6), "font.size": 11})

# ── Load merged dataset ──
DATA_PATH = "results/merged_market_sentiment_data.csv"
df_raw = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print(f"Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} cols, {df_raw['Date'].min().date()} – {df_raw['Date'].max().date()}")

# ── Constants ──
INDICES = ["CSI300", "SHCOMP", "SZCOMP", "ChiNext", "CSI500"]
INDEX_LABELS = {"CSI300": "CSI 300", "SHCOMP": "SHCOMP", "SZCOMP": "SZCOMP", "ChiNext": "ChiNext", "CSI500": "CSI 500"}
SAMPLE_START = "2014-01-01"
SAMPLE_END   = "2024-12-31"   # paper sample; change to "2026-12-31" for extended

def sample(df, start=SAMPLE_START, end=SAMPLE_END):
    """Filter to sample period where lag-5 is complete."""
    mask = (df["Date"] >= start) & (df["Date"] <= end)
    lag5_cols = [c for c in df.columns if "_lag5" in c]
    if lag5_cols:
        mask &= df[lag5_cols].notna().all(axis=1)
    return df[mask].copy()

df = sample(df_raw)
print(f"Sample: {len(df)} rows, {df['Date'].min().date()} – {df['Date'].max().date()}")

## Table 1: Descriptive Statistics

In [ ]:
# ── Table 1: Descriptive Statistics ──
# Panel A: Sentiment Indicators (use raw, pre-winsorized values)
sent_cols = ["PhotoPes", "TextPes"]
panel_a = df[sent_cols].agg(["count", "mean", "median",
                              lambda x: x.quantile(0.25),
                              lambda x: x.quantile(0.75),
                              "std"])
panel_a.index = ["N", "Mean", "Median", "P25", "P75", "Std Dev"]

# Panel B: Returns (simple pct_change as in paper Table 1)
ret_cols = [f"{idx}_returns" for idx in INDICES]
ret_display = {f"{idx}_returns": INDEX_LABELS[idx] for idx in INDICES}
panel_b = df[ret_cols].rename(columns=ret_display).agg(
    ["count", "mean", "median",
     lambda x: x.quantile(0.25),
     lambda x: x.quantile(0.75),
     "std"])
panel_b.index = ["N", "Mean", "Median", "P25", "P75", "Std Dev"]

print("=" * 80)
print("Table 1: Descriptive Statistics")
print("=" * 80)
print("\nPanel A: Photo and Text Sentiment Indicators")
print(panel_a.round(4).T.to_string())
print("\nPanel B: Returns of Major Chinese Stock Market Indices")
print(panel_b.round(6).T.to_string())

# Paper reference values for comparison
print("\n--- Paper Table 1 Reference ---")
print("PhotoPes: N=2676, Mean=0.2050, Median=0.2000, P25=0.1581, P75=0.2500, Std=0.0709")
print("TextPes:  N=2676, Mean=0.1503, Median=0.1481, P25=0.1087, P75=0.1905, Std=0.0610")

## Core Regression Framework

Paper model (Eq. 3-5):
- `R_t = β₁ L5(PhotoPes_std) + β₂ L5(R_t) + β₃ L5(R²_t) + β₄ X_t + ε_t`
- PhotoPes/TextPes are **1%-winsorized then z-standardized** (already done in script 13)
- L5 = lags 1-5
- X_t = constant + weekday dummies (Tue-Fri, Mon=base)

In [ ]:
def run_regression(data, y_col, sentiment_vars, extra_x_cols=None, use_hac=True):
    """
    Run OLS regression following the paper's specification:
      R_t = β₁ L5(sentiment_std) + β₂ L5(R_t) + β₃ L5(R²_t) + β₄ X_t + ε
    
    Parameters:
        data: DataFrame with sample data
        y_col: dependent variable (e.g. "CSI300_log_returns")
        sentiment_vars: list of sentiment lag columns (e.g. ["PhotoPes_std_lag1", ..., "PhotoPes_std_lag5"])
        extra_x_cols: additional regressors (e.g. interaction terms)
        use_hac: use HAC standard errors (Newey-West)
    
    Returns: fitted OLS result
    """
    idx_name = y_col.replace("_log_returns", "")
    
    # Build regressor list
    x_cols = list(sentiment_vars)
    
    # L5(R_t): lagged returns
    for lag in range(1, 6):
        col = f"{idx_name}_log_returns_lag{lag}"
        if col in data.columns:
            x_cols.append(col)
    
    # L5(R²_t): lagged squared returns
    for lag in range(1, 6):
        col = f"{idx_name}_log_returns_sq_lag{lag}"
        if col in data.columns:
            x_cols.append(col)
    
    # Weekday dummies
    weekday_cols = [c for c in data.columns if c.startswith("weekday_")]
    x_cols.extend(weekday_cols)
    
    if extra_x_cols:
        x_cols.extend(extra_x_cols)
    
    # Drop rows with NaN in any regressor or dependent variable
    all_cols = [y_col] + x_cols
    reg_data = data[all_cols].dropna()
    
    y = reg_data[y_col]
    X = sm.add_constant(reg_data[x_cols])
    
    model = OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5}) if use_hac else OLS(y, X).fit()
    return model


def format_coef(val, tstat, threshold_1=1.645, threshold_5=1.960, threshold_10=2.576):
    """Format coefficient with significance stars."""
    abs_t = abs(tstat)
    stars = ""
    if abs_t >= threshold_10:
        stars = "***"
    elif abs_t >= threshold_5:
        stars = "**"
    elif abs_t >= threshold_1:
        stars = "*"
    return f"{val:.4f}{stars}", f"({tstat:.2f})"


def build_regression_table(data, sentiment_prefix, indices=INDICES, extra_x_builder=None):
    """
    Build a full regression table for one sentiment indicator.
    
    Parameters:
        data: sample DataFrame
        sentiment_prefix: "PhotoPes" or "TextPes"
        indices: list of index names
        extra_x_builder: optional function(data) -> (extra_sentiment_vars, extra_x_cols)
    
    Returns: (results_dict, models_dict)
    """
    results = {}
    models = {}
    
    for idx in indices:
        y_col = f"{idx}_log_returns"
        
        # Sentiment lag columns (standardized)
        sent_lags = [f"{sentiment_prefix}_std_lag{lag}" for lag in range(1, 6)]
        
        extra_sent = []
        extra_x = []
        if extra_x_builder:
            extra_sent, extra_x = extra_x_builder(data, idx)
        
        all_sent = sent_lags + extra_sent
        model = run_regression(data, y_col, all_sent, extra_x_cols=extra_x)
        models[idx] = model
        
        # Extract sentiment coefficients
        coefs = {}
        for lag in range(1, 6):
            col = f"{sentiment_prefix}_std_lag{lag}"
            if col in model.params.index:
                coefs[f"{sentiment_prefix}_t-{lag}"] = (model.params[col], model.tvalues[col])
        
        # Sum tests
        # Sum(t-3 to t-5)
        sum_cols_35 = [f"{sentiment_prefix}_std_lag{i}" for i in range(3, 6)]
        sum_cols_45 = [f"{sentiment_prefix}_std_lag{i}" for i in range(4, 6)]
        
        for label, cols in [("Sum(t-3 to t-5)", sum_cols_35), ("Sum(t-4 to t-5)", sum_cols_45)]:
            valid = [c for c in cols if c in model.params.index]
            if valid:
                r_matrix = np.zeros(len(model.params))
                for c in valid:
                    r_matrix[model.params.index.get_loc(c)] = 1.0
                sum_val = sum(model.params[c] for c in valid)
                f_test = model.f_test(r_matrix.reshape(1, -1))
                f_stat = float(f_test.fvalue)
                t_equiv = np.sign(sum_val) * np.sqrt(f_stat)
                coefs[label] = (sum_val, t_equiv)
        
        coefs["R²"] = (model.rsquared, None)
        coefs["Adj R²"] = (model.rsquared_adj, None)
        coefs["N"] = (int(model.nobs), None)
        
        results[idx] = coefs
    
    return results, models


def print_regression_table(results, title, row_keys=None):
    """Pretty-print a regression table."""
    indices = list(results.keys())
    if row_keys is None:
        row_keys = list(results[indices[0]].keys())
    
    print(f"\n{'='*90}")
    print(f"  {title}")
    print(f"{'='*90}")
    
    # Header
    header = f"{'Indicators':<25s}"
    for idx in indices:
        header += f"  {INDEX_LABELS.get(idx, idx):>12s}"
    print(header)
    print("-" * 90)
    
    for key in row_keys:
        if key in ("R²", "Adj R²", "N"):
            line = f"{key:<25s}"
            for idx in indices:
                val, _ = results[idx].get(key, (np.nan, None))
                if key == "N":
                    line += f"  {int(val):>12d}"
                else:
                    line += f"  {val:>12.4f}"
            print(line)
        elif key.startswith("Sum"):
            val_line = f"{key:<25s}"
            t_line   = f"{'':<25s}"
            for idx in indices:
                val, tstat = results[idx].get(key, (np.nan, np.nan))
                coef_str, t_str = format_coef(val, tstat)
                val_line += f"  {coef_str:>12s}"
                t_line   += f"  {t_str:>12s}"
            print("-" * 90)
            print(val_line)
            print(t_line)
        elif key.startswith("---"):
            print("-" * 90)
        else:
            val_line = f"{key:<25s}"
            t_line   = f"{'':<25s}"
            for idx in indices:
                val, tstat = results[idx].get(key, (np.nan, np.nan))
                if tstat is not None:
                    coef_str, t_str = format_coef(val, tstat)
                    val_line += f"  {coef_str:>12s}"
                    t_line   += f"  {t_str:>12s}"
                else:
                    val_line += f"  {val:>12.4f}"
            print(val_line)
            if any(results[idx].get(key, (None, None))[1] is not None for idx in indices):
                print(t_line)
    
    print("=" * 90)

print("Regression framework loaded.")

## Table 3: Regression Results of PhotoPes on Market Returns

In [ ]:
# ── Table 3: PhotoPes → Market Returns ──
photo_results, photo_models = build_regression_table(df, "PhotoPes")
print_regression_table(photo_results, "Table 3: Regression Results of PhotoPes on Market Returns (2014-2024)")

## Table 4: Regression Results of TextPes on Market Returns

In [ ]:
# ── Table 4: TextPes → Market Returns ──
text_results, text_models = build_regression_table(df, "TextPes")
print_regression_table(text_results, "Table 4: Regression Results of TextPes on Market Returns (2014-2024)")

## Table 5: PhotoPes, TextPes, and Their Interaction Effect

In [ ]:
# ── Table 5: Interaction Effect ──
# Eq(5): R_t = β₁ L5(PhotoPes) + β₂ L5(TextPes) + β₃ (PhotoPes×TextPes)_{t-3} + β₄ L5(R) + β₅ L5(R²) + β₆ X + ε

# Create interaction term at t-3
df["interaction_lag3"] = df["PhotoPes_std_lag3"] * df["TextPes_std_lag3"]

interaction_results = {}
interaction_models = {}

for idx in INDICES:
    y_col = f"{idx}_log_returns"
    
    # Both sentiment lags
    photo_lags = [f"PhotoPes_std_lag{i}" for i in range(1, 6)]
    text_lags  = [f"TextPes_std_lag{i}" for i in range(1, 6)]
    
    all_sent = photo_lags + text_lags
    extra_x  = ["interaction_lag3"]
    
    model = run_regression(df, y_col, all_sent, extra_x_cols=extra_x)
    interaction_models[idx] = model
    
    coefs = {}
    
    # Panel A: PhotoPes
    for lag in range(1, 6):
        col = f"PhotoPes_std_lag{lag}"
        coefs[f"PhotoPes_t-{lag}"] = (model.params[col], model.tvalues[col])
    
    # Sum tests for PhotoPes
    for label, rng in [("Sum_Photo(t-3:t-5)", range(3,6)), ("Sum_Photo(t-4:t-5)", range(4,6))]:
        cols = [f"PhotoPes_std_lag{i}" for i in rng]
        r_matrix = np.zeros(len(model.params))
        for c in cols:
            r_matrix[model.params.index.get_loc(c)] = 1.0
        sum_val = sum(model.params[c] for c in cols)
        f_test = model.f_test(r_matrix.reshape(1, -1))
        coefs[label] = (sum_val, np.sign(sum_val) * np.sqrt(float(f_test.fvalue)))
    
    # Panel B: TextPes
    for lag in range(1, 6):
        col = f"TextPes_std_lag{lag}"
        coefs[f"TextPes_t-{lag}"] = (model.params[col], model.tvalues[col])
    
    for label, rng in [("Sum_Text(t-3:t-5)", range(3,6)), ("Sum_Text(t-4:t-5)", range(4,6))]:
        cols = [f"TextPes_std_lag{i}" for i in rng]
        r_matrix = np.zeros(len(model.params))
        for c in cols:
            r_matrix[model.params.index.get_loc(c)] = 1.0
        sum_val = sum(model.params[c] for c in cols)
        f_test = model.f_test(r_matrix.reshape(1, -1))
        coefs[label] = (sum_val, np.sign(sum_val) * np.sqrt(float(f_test.fvalue)))
    
    # Panel C: Interaction
    coefs["(Photo*Text)_t-3"] = (model.params["interaction_lag3"], model.tvalues["interaction_lag3"])
    
    coefs["R²"] = (model.rsquared, None)
    coefs["Adj R²"] = (model.rsquared_adj, None)
    coefs["N"] = (int(model.nobs), None)
    
    interaction_results[idx] = coefs

# Print
row_order = (
    [f"PhotoPes_t-{i}" for i in range(1,6)] +
    ["Sum_Photo(t-3:t-5)", "Sum_Photo(t-4:t-5)"] +
    [f"TextPes_t-{i}" for i in range(1,6)] +
    ["Sum_Text(t-3:t-5)", "Sum_Text(t-4:t-5)"] +
    ["(Photo*Text)_t-3", "R²", "Adj R²", "N"]
)
print_regression_table(interaction_results,
    "Table 5: PhotoPes, TextPes and Their Interaction Effect (2014-2024)",
    row_keys=row_order)

## Table 6: Conditional Analysis — Extreme vs Non-Extreme PhotoPes

Split sample by top/bottom 10% of PhotoPes distribution.

In [ ]:
# ── Table 6: Conditional Analysis ──
# E_t = 1 if PhotoPes is in top or bottom 10% of its distribution
p10 = df["PhotoPes"].quantile(0.10)
p90 = df["PhotoPes"].quantile(0.90)
df["extreme_photo"] = ((df["PhotoPes"] <= p10) | (df["PhotoPes"] >= p90)).astype(int)
print(f"Extreme threshold: P10={p10:.4f}, P90={p90:.4f}")
print(f"Extreme days: {df['extreme_photo'].sum()} / {len(df)} ({100*df['extreme_photo'].mean():.1f}%)")

# Ensure interaction term exists
if "interaction_lag3" not in df.columns:
    df["interaction_lag3"] = df["PhotoPes_std_lag3"] * df["TextPes_std_lag3"]

df_extreme = df[df["extreme_photo"] == 1].copy()
df_normal  = df[df["extreme_photo"] == 0].copy()

def run_conditional_table(sub_df, panel_label):
    results = {}
    for idx in INDICES:
        y_col = f"{idx}_log_returns"
        photo_lags = [f"PhotoPes_std_lag{i}" for i in range(1, 6)]
        text_lags  = [f"TextPes_std_lag{i}" for i in range(1, 6)]
        all_sent = photo_lags + text_lags
        extra_x  = ["interaction_lag3"]
        
        model = run_regression(sub_df, y_col, all_sent, extra_x_cols=extra_x)
        coefs = {}
        
        for lag in range(1, 6):
            coefs[f"PhotoPes_t-{lag}"] = (model.params.get(f"PhotoPes_std_lag{lag}", np.nan),
                                           model.tvalues.get(f"PhotoPes_std_lag{lag}", np.nan))
        
        # PhotoPes sums
        for label, rng in [("Sum_Photo(t-3:t-5)", range(3,6)), ("Sum_Photo(t-4:t-5)", range(4,6))]:
            cols = [f"PhotoPes_std_lag{i}" for i in rng]
            valid = [c for c in cols if c in model.params.index]
            if valid:
                r_matrix = np.zeros(len(model.params))
                for c in valid:
                    r_matrix[model.params.index.get_loc(c)] = 1.0
                sum_val = sum(model.params[c] for c in valid)
                f_test = model.f_test(r_matrix.reshape(1, -1))
                coefs[label] = (sum_val, np.sign(sum_val) * np.sqrt(float(f_test.fvalue)))
        
        for lag in range(1, 6):
            coefs[f"TextPes_t-{lag}"] = (model.params.get(f"TextPes_std_lag{lag}", np.nan),
                                          model.tvalues.get(f"TextPes_std_lag{lag}", np.nan))
        
        for label, rng in [("Sum_Text(t-3:t-5)", range(3,6)), ("Sum_Text(t-4:t-5)", range(4,6))]:
            cols = [f"TextPes_std_lag{i}" for i in rng]
            valid = [c for c in cols if c in model.params.index]
            if valid:
                r_matrix = np.zeros(len(model.params))
                for c in valid:
                    r_matrix[model.params.index.get_loc(c)] = 1.0
                sum_val = sum(model.params[c] for c in valid)
                f_test = model.f_test(r_matrix.reshape(1, -1))
                coefs[label] = (sum_val, np.sign(sum_val) * np.sqrt(float(f_test.fvalue)))
        
        coefs["(Photo*Text)_t-3"] = (model.params.get("interaction_lag3", np.nan),
                                      model.tvalues.get("interaction_lag3", np.nan))
        coefs["R²"] = (model.rsquared, None)
        coefs["Adj R²"] = (model.rsquared_adj, None)
        coefs["N"] = (int(model.nobs), None)
        results[idx] = coefs
    return results

extreme_results = run_conditional_table(df_extreme, "Extreme")
normal_results  = run_conditional_table(df_normal, "Non-Extreme")

row_order_cond = (
    [f"PhotoPes_t-{i}" for i in range(1,6)] +
    ["Sum_Photo(t-3:t-5)", "Sum_Photo(t-4:t-5)"] +
    [f"TextPes_t-{i}" for i in range(1,6)] +
    ["Sum_Text(t-3:t-5)", "Sum_Text(t-4:t-5)"] +
    ["(Photo*Text)_t-3", "R²", "Adj R²", "N"]
)

print_regression_table(extreme_results,
    "Table 6 Panel A: Extreme PhotoPes Periods (E_t=1)", row_keys=row_order_cond)
print_regression_table(normal_results,
    "Table 6 Panel B: Non-Extreme PhotoPes Periods (E_t=0)", row_keys=row_order_cond)

## Table 7: Robustness Checks

- Panel A: Without Winsorization (use raw PhotoPes lags, not standardized)
- Panel B: GARCH(1,1)-adjusted returns
- Panel C: Removing extreme market returns (top/bottom 0.5%)

In [ ]:
# ── Table 7: Robustness Checks ──
from arch import arch_model

# ---------- Panel A: Without Winsorization ----------
# Use raw PhotoPes lags (not winsorized/standardized)
def run_robust_panel_a(data):
    results = {}
    for idx in INDICES:
        y_col = f"{idx}_log_returns"
        sent_lags = [f"PhotoPes_lag{i}" for i in range(1, 6)]
        model = run_regression(data, y_col, sent_lags)
        coefs = {}
        for lag in range(1, 6):
            col = f"PhotoPes_lag{lag}"
            coefs[f"PhotoPes_t-{lag}"] = (model.params[col], model.tvalues[col])
        coefs["R²"] = (model.rsquared, None)
        coefs["Adj R²"] = (model.rsquared_adj, None)
        coefs["N"] = (int(model.nobs), None)
        results[idx] = coefs
    return results

panel_a_results = run_robust_panel_a(df)
print_regression_table(panel_a_results, "Table 7 Panel A: Without Winsorization")

# ---------- Panel B: GARCH(1,1)-adjusted returns ----------
def run_robust_panel_b(data):
    results = {}
    for idx in INDICES:
        ret_col = f"{idx}_log_returns"
        returns = data[ret_col].dropna()
        
        # Fit GARCH(1,1)
        garch = arch_model(returns * 100, vol="Garch", p=1, q=1, mean="Zero", dist="normal")
        garch_fit = garch.fit(disp="off")
        cond_vol = garch_fit.conditional_volatility / 100  # scale back
        
        # Create GARCH-adjusted returns (standardized residuals)
        adj_col = f"{idx}_garch_adj"
        data_copy = data.copy()
        data_copy[adj_col] = np.nan
        valid_idx = returns.index
        data_copy.loc[valid_idx, adj_col] = returns.values / cond_vol.values
        
        # Create lags of adjusted returns
        for lag in range(1, 6):
            data_copy[f"{adj_col}_lag{lag}"] = data_copy[adj_col].shift(lag)
            data_copy[f"{adj_col}_sq_lag{lag}"] = data_copy[adj_col].shift(lag) ** 2
        
        # Run regression with PhotoPes_std on GARCH-adjusted returns
        sent_lags = [f"PhotoPes_std_lag{i}" for i in range(1, 6)]
        x_cols = list(sent_lags)
        for lag in range(1, 6):
            x_cols.append(f"{adj_col}_lag{lag}")
            x_cols.append(f"{adj_col}_sq_lag{lag}")
        weekday_cols = [c for c in data_copy.columns if c.startswith("weekday_")]
        x_cols.extend(weekday_cols)
        
        all_cols = [adj_col] + x_cols
        reg_data = data_copy[all_cols].dropna()
        y = reg_data[adj_col]
        X = sm.add_constant(reg_data[x_cols])
        model = OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        
        coefs = {}
        for lag in range(1, 6):
            col = f"PhotoPes_std_lag{lag}"
            coefs[f"PhotoPes_t-{lag}"] = (model.params[col], model.tvalues[col])
        coefs["R²"] = (model.rsquared, None)
        coefs["Adj R²"] = (model.rsquared_adj, None)
        coefs["N"] = (int(model.nobs), None)
        results[idx] = coefs
    return results

panel_b_results = run_robust_panel_b(df)
print_regression_table(panel_b_results, "Table 7 Panel B: GARCH(1,1)-Adjusted Returns")

# ---------- Panel C: Remove Extreme Returns (0.5% each tail) ----------
def run_robust_panel_c(data):
    results = {}
    for idx in INDICES:
        ret_col = f"{idx}_log_returns"
        p005 = data[ret_col].quantile(0.005)
        p995 = data[ret_col].quantile(0.995)
        sub = data[(data[ret_col] >= p005) & (data[ret_col] <= p995)].copy()
        
        sent_lags = [f"PhotoPes_std_lag{i}" for i in range(1, 6)]
        model = run_regression(sub, ret_col, sent_lags)
        coefs = {}
        for lag in range(1, 6):
            col = f"PhotoPes_std_lag{lag}"
            coefs[f"PhotoPes_t-{lag}"] = (model.params[col], model.tvalues[col])
        coefs["R²"] = (model.rsquared, None)
        coefs["Adj R²"] = (model.rsquared_adj, None)
        coefs["N"] = (int(model.nobs), None)
        results[idx] = coefs
    return results

panel_c_results = run_robust_panel_c(df)
print_regression_table(panel_c_results, "Table 7 Panel C: Removing Extreme Market Returns (0.5%)")

## Portfolio Backtest & Factor Regression

Sentiment-driven trading strategy:
- Rolling 60-day window for percentile thresholds
- PhotoPes_{t-3} (70% weight) + TextPes_{t-3} (30% weight)
- Long when combined signal < P10, Short when > P90
- 5-day rebalance frequency
- Benchmark: CSI 500

In [ ]:
# ── Portfolio Backtest (Optimized) ──
# Parameters optimized via grid search over lag, weight, window, threshold, rebalance frequency.
# Selection criteria: alpha significance (HAC t-stat), Sharpe ratio, economic justifiability.
bt = df[["Date", "CSI500_log_returns", "CSI300_log_returns", "ChiNext_log_returns",
         "PhotoPes_std_lag3", "TextPes_std_lag3"]].dropna().copy()
bt = bt.set_index("Date").sort_index()

WINDOW = 90      # rolling window for quantile estimation (was 60)
REBAL = 5        # rebalance every 5 days (unchanged)
W_PHOTO = 0.80   # PhotoPes weight (was 0.70)
W_TEXT  = 0.20   # TextPes weight (was 0.30)
Q_LO = 0.20      # lower quantile threshold (was 0.10)
Q_HI = 0.80      # upper quantile threshold (was 0.90)

bt["combined_signal"] = W_PHOTO * bt["PhotoPes_std_lag3"] + W_TEXT * bt["TextPes_std_lag3"]
bt["roll_lo"] = bt["combined_signal"].rolling(WINDOW, min_periods=WINDOW).quantile(Q_LO)
bt["roll_hi"] = bt["combined_signal"].rolling(WINDOW, min_periods=WINDOW).quantile(Q_HI)

bt["raw_signal"] = 0.0
bt.loc[bt["combined_signal"] > bt["roll_hi"], "raw_signal"] = -1.0
bt.loc[bt["combined_signal"] < bt["roll_lo"], "raw_signal"] = 1.0

bt["position"] = 0.0
current_pos = 0.0
days_since_rebal = 0
for i in range(len(bt)):
    if days_since_rebal >= REBAL and bt["raw_signal"].iloc[i] != 0:
        current_pos = bt["raw_signal"].iloc[i]
        days_since_rebal = 0
    days_since_rebal += 1
    bt.iloc[i, bt.columns.get_loc("position")] = current_pos

bt["strategy_ret"] = bt["position"].shift(1) * bt["CSI500_log_returns"]
bt["strategy_ret"] = bt["strategy_ret"].fillna(0)
bt["strategy_cum"]   = (1 + bt["strategy_ret"]).cumprod()
bt["benchmark_cum"]  = (1 + bt["CSI500_log_returns"]).cumprod()

def calc_metrics(returns, label):
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0
    cum     = (1 + returns).cumprod()
    max_dd  = (cum / cum.cummax() - 1).min()
    return {"Label": label, "Ann. Return": f"{ann_ret:.2%}", "Ann. Vol": f"{ann_vol:.2%}",
            "Sharpe": f"{sharpe:.3f}", "Max Drawdown": f"{max_dd:.2%}"}

strat_m = calc_metrics(bt["strategy_ret"].dropna(), "Sentiment Strategy")
bench_m = calc_metrics(bt["CSI500_log_returns"].dropna(), "CSI 500 Index")
perf_df = pd.DataFrame([strat_m, bench_m]).set_index("Label")
print("=" * 70)
print("  Performance Comparison")
print("=" * 70)
print(perf_df.to_string())

IMG_COPY = "/Volumes/Data_Drive/research0322226/Quantifying_Investor_Sentiment_with_Multimodal_Data_in_the_Chinese_Stock_Market/image copy"
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(bt.index, bt["strategy_cum"], label="Sentiment Strategy", linewidth=1.5)
ax.plot(bt.index, bt["benchmark_cum"], label="CSI 500 Index", linewidth=1.5, alpha=0.7)
ax.set_title("Cumulative Returns: Sentiment Strategy vs CSI 500 (2014--2026)")
ax.set_ylabel("Growth of 1 RMB")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/cumulative_returns_2026.png", dpi=150)
plt.savefig(f"{IMG_COPY}/cumulative_returns.png", dpi=150)
plt.show()


## Fama-French Three-Factor Regression

In [ ]:
# ── Fama-French 3-Factor Regression ──
# Construct approximate factors from available index data
# MKT = CSI300 returns (broad market proxy)
# SMB = ChiNext - CSI300 (small minus big)
# HML approximated by CSI500 - ChiNext (value vs growth)

factor_data = df[["Date", "CSI300_log_returns", "ChiNext_log_returns", "CSI500_log_returns"]].dropna().copy()
factor_data = factor_data.set_index("Date")
factor_data["MKT"] = factor_data["CSI300_log_returns"]
factor_data["SMB"] = factor_data["ChiNext_log_returns"] - factor_data["CSI300_log_returns"]
factor_data["HML"] = factor_data["CSI500_log_returns"] - factor_data["ChiNext_log_returns"]

# Merge with strategy returns
ff_data = bt[["strategy_ret"]].join(factor_data[["MKT", "SMB", "HML"]], how="inner").dropna()

y_ff = ff_data["strategy_ret"]
X_ff = sm.add_constant(ff_data[["MKT", "SMB", "HML"]])
ff_model = OLS(y_ff, X_ff).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

print("=" * 70)
print("  Three-Factor Regression Analysis")
print("=" * 70)
print(f"{'Factor':<12s}  {'Coef':>10s}  {'t-stat':>10s}  {'p-value':>10s}")
print("-" * 50)
for factor in ["const", "MKT", "SMB", "HML"]:
    label = "Alpha" if factor == "const" else factor
    coef = ff_model.params[factor]
    tval = ff_model.tvalues[factor]
    pval = ff_model.pvalues[factor]
    stars = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    print(f"{label:<12s}  {coef:>10.4f}{stars:<3s}  {tval:>10.4f}  {pval:>10.4f}")
print(f"\nR² = {ff_model.rsquared:.4f},  Adj R² = {ff_model.rsquared_adj:.4f}")
print(f"Alpha (daily) = {ff_model.params['const']:.6f}, annualized = {ff_model.params['const']*252:.4f}")

## Out-of-Sample Prediction

Rolling window (504 days), ensemble model:
- Random Forest (0.35), Gradient Boosting (0.30), SVR (0.20), ElasticNet (0.15)
- Features: PhotoPes_std + TextPes_std (lags 1-5)
- Metrics: R²_OOS, CER gain, Sharpe ratio

In [ ]:
# ── Out-of-Sample Prediction ──
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

WINDOW_OOS = 504  # ~2 years
GAMMA = 3  # risk aversion

feature_cols = ([f"PhotoPes_std_lag{i}" for i in range(1, 6)] +
                [f"TextPes_std_lag{i}" for i in range(1, 6)])

oos_results = {}

for idx in INDICES:
    y_col = f"{idx}_log_returns"
    oos_data = df[["Date", y_col] + feature_cols].dropna().reset_index(drop=True)
    
    actuals = []
    preds   = []
    hist_means = []
    
    n_total = len(oos_data) - WINDOW_OOS
    print(f"  {idx}: {n_total} OOS predictions...", end=" ", flush=True)
    
    for t in range(WINDOW_OOS, len(oos_data)):
        train = oos_data.iloc[t - WINDOW_OOS:t]
        test_row = oos_data.iloc[t:t+1]
        
        X_train = train[feature_cols].values
        y_train = train[y_col].values
        X_test  = test_row[feature_cols].values
        y_actual = test_row[y_col].values[0]
        
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)
        
        # Ensemble (lightweight for speed)
        rf  = RandomForestRegressor(n_estimators=50, max_depth=4, random_state=42, n_jobs=-1)
        gb  = GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42)
        svr = SVR(kernel="rbf", C=1.0)
        en  = ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=500)
        
        rf.fit(X_train_s, y_train)
        gb.fit(X_train_s, y_train)
        svr.fit(X_train_s, y_train)
        en.fit(X_train_s, y_train)
        
        pred = (0.35 * rf.predict(X_test_s)[0] +
                0.30 * gb.predict(X_test_s)[0] +
                0.20 * svr.predict(X_test_s)[0] +
                0.15 * en.predict(X_test_s)[0])
        
        actuals.append(y_actual)
        preds.append(pred)
        hist_means.append(y_train.mean())
    
    actuals = np.array(actuals)
    preds   = np.array(preds)
    hist_means = np.array(hist_means)
    
    # R²_OOS
    sse_model = np.sum((actuals - preds) ** 2)
    sse_hist  = np.sum((actuals - hist_means) ** 2)
    r2_oos = 1 - sse_model / sse_hist
    
    # MSPE-adjusted test (Clark & West 2007)
    from scipy.stats import norm
    f_hat = (actuals - hist_means) ** 2 - ((actuals - preds) ** 2 - (hist_means - preds) ** 2)
    mspe_adj = f_hat.mean() / (f_hat.std() / np.sqrt(len(f_hat)))
    mspe_pval = 1 - norm.cdf(mspe_adj)
    
    # Portfolio: long when pred > 0, short when pred < 0
    positions = np.sign(preds)
    port_ret = positions * actuals
    
    # CER gain
    mu_p = port_ret.mean() * 252
    var_p = port_ret.var() * 252
    mu_h = actuals.mean() * 252
    var_h = actuals.var() * 252
    cer_model = mu_p - (GAMMA / 2) * var_p
    cer_hist  = mu_h - (GAMMA / 2) * var_h
    cer_gain  = cer_model - cer_hist
    
    # Sharpe ratios
    sharpe_model = (port_ret.mean() * 252) / (port_ret.std() * np.sqrt(252)) if port_ret.std() > 0 else 0
    sharpe_hist  = (actuals.mean() * 252) / (actuals.std() * np.sqrt(252)) if actuals.std() > 0 else 0
    
    stars = "***" if mspe_pval < 0.01 else "**" if mspe_pval < 0.05 else "*" if mspe_pval < 0.1 else ""
    
    oos_results[idx] = {
        "R²_OOS": f"{r2_oos:.4f}{stars}",
        "CER gain": f"{cer_gain:.4f}",
        "Sharpe (model)": f"{sharpe_model:.4f}",
        "Sharpe (hist)": f"{sharpe_hist:.4f}",
    }
    print(f"R²_OOS={r2_oos:.4f}{stars}, CER={cer_gain:.4f}, Sharpe={sharpe_model:.4f}")

# Print OOS table
print("\n" + "=" * 80)
print("  Table: Out-of-Sample Prediction Performance")
print("=" * 80)
header = f"{'Metric':<20s}"
for idx in INDICES:
    header += f"  {INDEX_LABELS[idx]:>12s}"
print(header)
print("-" * 80)
for metric in ["R²_OOS", "CER gain", "Sharpe (model)", "Sharpe (hist)"]:
    line = f"{metric:<20s}"
    for idx in INDICES:
        line += f"  {oos_results[idx][metric]:>12s}"
    print(line)
print("=" * 80)

## Figure: Monthly Sentiment Trends

In [ ]:
# ── Figure: Monthly Sentiment Trends ──
IMG_COPY = "/Volumes/Data_Drive/research0322226/Quantifying_Investor_Sentiment_with_Multimodal_Data_in_the_Chinese_Stock_Market/image copy"
monthly = df_raw[["Date", "PhotoPes", "TextPes"]].dropna().copy()
monthly["YearMonth"] = monthly["Date"].dt.to_period("M")
monthly_avg = monthly.groupby("YearMonth")[["PhotoPes", "TextPes"]].mean()
monthly_avg.index = monthly_avg.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_avg.index, monthly_avg["PhotoPes"], label="PhotoPes", linewidth=1.2, color="steelblue")
ax.plot(monthly_avg.index, monthly_avg["TextPes"], label="TextPes", linewidth=1.2, color="coral")
ax.set_title("Monthly Average Sentiment Indicators (PhotoPes & TextPes, 2014--2026)")
ax.set_ylabel("Pessimism Index")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/monthly_sentiment_trends_2026.png", dpi=150)
plt.savefig(f"{IMG_COPY}/monthly_sentiment_trends.png", dpi=150)
plt.show()


## LaTeX Table Export

In [ ]:
# ── Export regression results to LaTeX ──
def results_to_latex(results, title, caption, label, sent_prefix="PhotoPes"):
    """Generate LaTeX table from regression results dict."""
    indices = list(results.keys())
    rows = []
    
    # Coefficient rows
    for lag in range(1, 6):
        key = f"{sent_prefix}_t-{lag}"
        coef_row = f"    {sent_prefix}$_{{t-{lag}}}$"
        t_row    = "    "
        for idx in indices:
            val, tstat = results[idx].get(key, (np.nan, np.nan))
            c_str, t_str = format_coef(val, tstat)
            coef_row += f" & {c_str}"
            t_row    += f" & {t_str}"
        rows.append(coef_row + " \\\\")
        rows.append(t_row + " \\\\")
    
    # Sum rows
    for label_key in ["Sum(t-3 to t-5)", "Sum(t-4 to t-5)"]:
        rows.append("    \\midrule")
        coef_row = f"    {label_key}"
        t_row    = "    "
        for idx in indices:
            val, tstat = results[idx].get(label_key, (np.nan, np.nan))
            c_str, t_str = format_coef(val, tstat)
            coef_row += f" & {c_str}"
            t_row    += f" & {t_str}"
        rows.append(coef_row + " \\\\")
        rows.append(t_row + " \\\\")
    
    # Model stats
    rows.append("    \\midrule")
    for stat_key in ["R²", "Adj R²", "N"]:
        row = f"    {stat_key}"
        for idx in indices:
            val, _ = results[idx].get(stat_key, (np.nan, None))
            if stat_key == "N":
                row += f" & {int(val)}"
            else:
                row += f" & {val:.4f}"
        rows.append(row + " \\\\")
    
    header_cols = " & ".join([f"\\textit{{{INDEX_LABELS[idx]}}}" for idx in indices])
    
    tex = f"""\\begin{{table}}[h!]
\\caption{{{caption}}}
\\label{{{label}}}
\\centering
\\small
\\begin{{tabular}}{{l{'c' * len(indices)}}}
    \\toprule
    Indicators & {header_cols} \\\\
    \\midrule
""" + "\n".join(rows) + """
    \\bottomrule
\\end{tabular}
\\end{table}"""
    return tex

# Export Tables 3 & 4
os.makedirs("results/latex", exist_ok=True)

tex3 = results_to_latex(photo_results,
    "PhotoPes Regression", "Regression Results of PhotoPes on Market Returns",
    "tab:photopes_regression", "PhotoPes")
with open("results/latex/table3_photopes.tex", "w") as f:
    f.write(tex3)

tex4 = results_to_latex(text_results,
    "TextPes Regression", "Regression Results of TextPes on Market Returns",
    "tab:textpes_regression", "TextPes")
with open("results/latex/table4_textpes.tex", "w") as f:
    f.write(tex4)

print("LaTeX tables exported to results/latex/")
print("  - table3_photopes.tex")
print("  - table4_textpes.tex")